In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
uploaded = files.upload()

Saving orders_enriched.csv to orders_enriched.csv


In [21]:
df=pd.read_csv("orders_enriched.csv")

## 1. KHẢO SÁT DỮ LIỆU GỐC (DATA PROFILING)

In [22]:
# 1.1 Kiểm tra kích thước (shape), các cột, kiểu dữ liệu hiện tại
print("1.1. Thông tin tổng quan về DataFrame:")
display(df.info())

1.1. Thông tin tổng quan về DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   order_id             646945 non-null  int64 
 1   order_date           646945 non-null  object
 2   customer_id          646945 non-null  int64 
 3   zip                  646945 non-null  int64 
 4   city                 646945 non-null  object
 5   region               646945 non-null  object
 6   district             646945 non-null  object
 7   order_status         646945 non-null  object
 8   payment_method       646945 non-null  object
 9   device_type          646945 non-null  object
 10  order_source         646945 non-null  object
 11  sales_employee_id    646945 non-null  object
 12  sales_employee_name  646945 non-null  object
 13  marital_status       646945 non-null  object
 14  education_level      646945 non-null  object


None

In [23]:
# 1.2 Thống kê tỷ lệ khuyết thiếu/null trên từng cột
print("\n1.2. Thống kê giá trị khuyết thiếu (null) trên từng cột:")
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100
null_info = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage': null_percentages
})
display(null_info.sort_values(by='Null Percentage', ascending=False))


1.2. Thống kê giá trị khuyết thiếu (null) trên từng cột:


,Null Count,Null Percentage
order_id,0,0.0
order_date,0,0.0
customer_id,0,0.0
zip,0,0.0
city,0,0.0
region,0,0.0
district,0,0.0
order_status,0,0.0
payment_method,0,0.0
device_type,0,0.0


In [24]:
# 1.3 Kiểm tra số lượng dòng trùng lặp (duplicates)
print("\n1.3. Số lượng dòng trùng lặp:")
duplicate_rows = df.duplicated().sum()
print(f"Có {duplicate_rows} dòng bị trùng lặp.")

# Giả định 'order_id' là khóa chính (Primary Key) và kiểm tra tính duy nhất
print("\nKiểm tra tính duy nhất của khóa chính (Primary Key - 'order_id'):")
is_pk_unique = df['order_id'].nunique() == len(df)
print(f"'order_id' có phải là khóa chính duy nhất không? {is_pk_unique}")
if not is_pk_unique:
    print(f"Số lượng giá trị duy nhất của 'order_id': {df['order_id'].nunique()}")
    print(f"Tổng số dòng: {len(df)}")


1.3. Số lượng dòng trùng lặp:
Có 0 dòng bị trùng lặp.

Kiểm tra tính duy nhất của khóa chính (Primary Key - 'order_id'):
'order_id' có phải là khóa chính duy nhất không? True


In [25]:
# 1.4 Kiểm tra các giá trị phân loại (.unique()) và khoảng giá trị số học cơ bản.
print("\n1.4. Kiểm tra các giá trị phân loại và thống kê cơ bản cho cột số:")
for column in df.columns:
    if df[column].dtype == 'object': # Giả định các cột object là phân loại hoặc chuỗi
        print(f"\nCột '{column}' (Phân loại/Chuỗi): {df[column].nunique()} giá trị duy nhất")
        if df[column].nunique() < 50: # Hiển thị unique values nếu số lượng không quá lớn
            display(df[column].unique())
        else:
            print(f"  Có quá nhiều giá trị duy nhất ({df[column].nunique()}) để hiển thị toàn bộ.")
    elif pd.api.types.is_numeric_dtype(df[column]):
        print(f"\nCột '{column}' (Số học):")
        display(df[column].describe())


1.4. Kiểm tra các giá trị phân loại và thống kê cơ bản cho cột số:

Cột 'order_id' (Số học):


,order_id
count,646945.000000
mean,417189.470332
std,240785.704463
min,1.000000
25%,208728.000000
50%,417211.000000
75%,625628.000000
max,834397.000000



Cột 'order_date' (Phân loại/Chuỗi): 3833 giá trị duy nhất
  Có quá nhiều giá trị duy nhất (3833) để hiển thị toàn bộ.

Cột 'customer_id' (Số học):


,customer_id
count,646945.000000
mean,84906.203535
std,48446.922752
min,1.000000
25%,41336.000000
50%,87279.000000
75%,133282.000000
max,157563.000000



Cột 'zip' (Số học):


...

array(['Đại học', 'Sau đại học', 'Cao đẳng', 'Trung cấp'], dtype=object)


Cột 'years_experience' (Số học):


,years_experience
count,646945.000000
mean,10.907317
std,5.471233
min,1.000000
25%,7.000000
50%,11.000000
75%,16.000000
max,20.000000



Cột 'comment' (Phân loại/Chuỗi): 18 giá trị duy nhất


array(['Dịch vụ chăm sóc khách hàng chuyên nghiệp, thái độ thân thiện và lịch sự.',
       'Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ trợ khi cần.',
       'Khách hàng không có thêm phản hồi về chất lượng hỗ trợ.',
       'Phản hồi thắc mắc đầy đủ, hỗ trợ xuyên suốt quá trình đặt hàng.',
       'Khách hàng đánh giá mức độ hài lòng ở mức khá.',
       'Tư vấn chi tiết, thái độ chuyên nghiệp và thân thiện.',
       'Hỗ trợ nhanh chóng, tư vấn rõ ràng, khách hàng hài lòng với trải nghiệm.',
       'Quá trình hỗ trợ diễn ra bình thường, đúng quy trình.',
       'Nhân viên giao tiếp tốt, xử lý yêu cầu đúng hẹn và tận tâm.',
       'Chăm sóc khách hàng tốt, thời gian phản hồi ngắn.',
       'Khách hàng đánh giá cao sự chủ động và trách nhiệm của bộ phận hỗ trợ.',
       'Khách hàng đánh giá nhân viên hỗ trợ rất nhiệt tình, phản hồi nhanh và giải quyết vấn đề hiệu quả.',
       'Cần cải thiện tốc độ hỗ trợ vào giờ cao điểm.',
       'Giải quyết khiếu nại nhanh, khách hàng hài lòng với kết q

## 2. LÀM SẠCH VÀ CHUẨN HÓA (CLEANING & TRANSFORMATION) & Tách bảng dữ liệu

In [26]:
# Chuẩn hóa kiểu dữ liệu cho cột 'order_date'
original_dtype = df['order_date'].dtype
df['order_date'] = pd.to_datetime(df['order_date'])
print(f"Cột 'order_date' đã được chuyển đổi từ {original_dtype} sang {df['order_date'].dtype}")


Cột 'order_date' đã được chuyển đổi từ object sang datetime64[ns]


In [27]:


# 2.1 Tạo DataFrame 'sales_employee' với các cột được yêu cầu
sales_employee_df = df[['sales_employee_id', 'sales_employee_name', 'marital_status', 'education_level', 'years_experience']].copy()

# Loại bỏ các hàng trùng lặp để sales_employee_id là duy nhất (Primary Key)
sales_employee_df = sales_employee_df.drop_duplicates(subset=['sales_employee_id'])

print("Bảng SALES EMPLOYEE đã được tạo:")
display(sales_employee_df.head())
print(f"Số lượng nhân viên (PK): {len(sales_employee_df)}")

Bảng SALES EMPLOYEE đã được tạo:


,sales_employee_id,sales_employee_name,marital_status,education_level,years_experience
0,EMP0103,Nguyễn Thị Phúc,Đã kết hôn,Đại học,20
1,EMP0180,Lê Gia Hùng,Đã kết hôn,Sau đại học,3
2,EMP0093,Bùi Minh Quân,Đã kết hôn,Cao đẳng,18
3,EMP0015,Võ Quốc Mai,Độc thân,Trung cấp,12
4,EMP0107,Hồ Thanh Yến,Độc thân,Đại học,17


Số lượng nhân viên (PK): 200


In [28]:
# 2.2 Tạo lại DataFrame 'order' với các cột cụ thể theo yêu cầu
# Cấu trúc: order_id, customer_id, sales_employee_id, order_date, order_source, order_status, device_type, comment, zip
order_cols = ['order_id', 'customer_id', 'sales_employee_id', 'order_date', 'order_source', 'order_status', 'device_type', 'comment', 'zip']
order_df = df[order_cols].copy()

print("Bảng ORDER đã được tạo lại:")
display(order_df.head())
print(f"Tổng số đơn hàng: {len(order_df)}")

Bảng ORDER đã được tạo lại:


,order_id,customer_id,sales_employee_id,order_date,order_source,order_status,device_type,comment,zip
0,1,58578,EMP0103,2012-07-04,paid_search,delivered,desktop,"Dịch vụ chăm sóc khách hàng chuyên nghiệp, thá...",1109
1,2,58621,EMP0180,2012-07-04,paid_search,returned,mobile,"Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ tr...",1330
2,3,58811,EMP0093,2012-07-04,direct,delivered,desktop,Khách hàng không có thêm phản hồi về chất lượn...,1473
3,4,59453,EMP0015,2012-07-04,referral,delivered,desktop,"Phản hồi thắc mắc đầy đủ, hỗ trợ xuyên suốt qu...",2360
4,6,57821,EMP0107,2012-07-06,email_campaign,delivered,mobile,Khách hàng đánh giá mức độ hài lòng ở mức khá.,2886


Tổng số đơn hàng: 646945


## 3. KIỂM TRA TÍNH TOÀN VẸN (DATA QUALITY ASSURANCE)

In [29]:
# 3.1 Khóa chính không null và không trùng lặp (đã kiểm tra ở bước 1.3, tái khẳng định)

assert df['order_id'].isnull().sum() == 0, "Lỗi: Khóa chính 'order_id' chứa giá trị null."
assert df['order_id'].nunique() == len(df), "Lỗi: Khóa chính 'order_id' chứa giá trị trùng lặp."

print("Kiểm tra khóa chính 'order_id': OK")


Kiểm tra khóa chính 'order_id': OK


In [30]:
# 3.2 Logic thời gian hợp lệ (ví dụ: order_date phải là ngày hợp lệ và trong khoảng thời gian nhất định)

# Kiểm tra order_date không có NaT (Not a Time) sau chuyển đổi
assert df['order_date'].isnull().sum() == 0, "Lỗi: Cột 'order_date' chứa giá trị không hợp lệ (NaT)."

# Kiểm tra ngày tháng nằm trong một khoảng hợp lý (ví dụ: sau năm 2000)
# Có thể điều chỉnh khoảng thời gian này dựa trên nghiệp vụ cụ thể
assert (df['order_date'] >= pd.to_datetime('2000-01-01')).all(), "Lỗi: Cột 'order_date' chứa ngày quá khứ không hợp lệ."
assert (df['order_date'] <= pd.to_datetime('today')).all(), "Lỗi: Cột 'order_date' chứa ngày trong tương lai không hợp lệ."

print("Kiểm tra logic thời gian 'order_date': OK")

Kiểm tra logic thời gian 'order_date': OK


In [31]:
# 3.3 Ràng buộc số học hợp lệ

# Kiểm tra 'zip', 'customer_id' là số dương
assert (df['zip'] > 0).all(), "Lỗi: Cột 'zip' chứa giá trị không hợp lệ (nhỏ hơn hoặc bằng 0)."
assert (df['customer_id'] > 0).all(), "Lỗi: Cột 'customer_id' chứa giá trị không hợp lệ (nhỏ hơn hoặc bằng 0)."

# Kiểm tra 'years_experience' không âm
assert (df['years_experience'] >= 0).all(), "Lỗi: Cột 'years_experience' chứa giá trị âm."

print("Kiểm tra ràng buộc số học: OK")

Kiểm tra ràng buộc số học: OK


In [32]:
# 3.4 Các giá trị phân loại (categorical) phải khớp với miền giá trị hợp lệ

# Vì không có Data Dictionary cụ thể, chúng ta sẽ kiểm tra các giá trị không rỗng/null cho các cột object.
# Nếu có Data Dictionary, bạn sẽ thay thế bằng các danh sách giá trị hợp lệ cụ thể.

categorical_cols = [
    'city', 'region', 'district', 'order_status', 'payment_method',
    'device_type', 'order_source', 'marital_status', 'education_level'
]

for col in categorical_cols:
    assert df[col].isnull().sum() == 0, f"Lỗi: Cột phân loại '{col}' chứa giá trị null."
    assert (df[col] != '').all(), f"Lỗi: Cột phân loại '{col}' chứa giá trị rỗng."

    # Ví dụ kiểm tra với một danh sách giá trị hợp lệ (nếu có Data Dictionary)
    # if col == 'order_status':
    #     valid_statuses = ['Completed', 'Pending', 'Cancelled']
    #     assert df[col].isin(valid_statuses).all(), f"Lỗi: Cột '{col}' chứa giá trị không hợp lệ."

print("Kiểm tra giá trị phân loại: OK (dựa trên giả định không rỗng/null)")


Kiểm tra giá trị phân loại: OK (dựa trên giả định không rỗng/null)


## 4. KIỂM TRA KHÓA NGOẠI (FOREIGN KEY VALIDATION)
Bước này đảm bảo mối quan hệ giữa bảng `order_df` và `sales_employee_df` thông qua cột `sales_employee_id` là chính xác.

In [33]:
print("--- BẮT ĐẦU KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU ---\n")

# 1. Kiểm tra Khóa Chính (Primary Keys)
def check_pk(df, col_name, table_name):
    is_unique = df[col_name].is_unique
    is_not_null = df[col_name].notnull().all()
    print(f"Bảng {table_name} - Cột {col_name} (PK):")
    print(f"  - Duy nhất: {'PASS' if is_unique else 'FAIL'}")
    print(f"  - Không Null: {'PASS' if is_not_null else 'FAIL'}")
    assert is_unique and is_not_null, f"Lỗi PK tại bảng {table_name}"

check_pk(order_df, 'order_id', 'ORDER')
check_pk(sales_employee_df, 'sales_employee_id', 'SALES EMPLOYEE')

# 2. Kiểm tra Khóa Ngoại (Foreign Keys)
print("\n--- Kiểm tra Khóa Ngoại ---")

# FK: order.sales_employee_id -> sales_employee.sales_employee_id
orphaned_emp = set(order_df['sales_employee_id']) - set(sales_employee_df['sales_employee_id'])
if not orphaned_emp:
    print("FK sales_employee_id: PASS (Tất cả tham chiếu đều hợp lệ)")
else:
    print(f"FK sales_employee_id: FAIL (Tìm thấy {len(orphaned_emp)} ID không tồn tại)")
    assert not orphaned_emp, "Lỗi vi phạm ràng buộc khóa ngoại nhân viên"

# 3. Kiểm tra tính hợp lệ của các cột FK khác (customer_id, zip)
# Vì chưa tách bảng Customer hay Geography, ta kiểm tra logic cơ bản
print("\n--- Kiểm tra Logic Khóa Phụ khác ---")
for fk_col in ['customer_id', 'zip']:
    has_null = order_df[fk_col].isnull().any()
    is_positive = (order_df[fk_col].astype(float) > 0).all()
    print(f"Cột {fk_col} (FK dự kiến):")
    print(f"  - Không Null: {'PASS' if not has_null else 'FAIL'}")
    print(f"  - Giá trị dương: {'PASS' if is_positive else 'FAIL'}")

print("\n=> TẤT CẢ KIỂM TRA HOÀN TẤT HỢP LỆ.")

--- BẮT ĐẦU KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU ---

Bảng ORDER - Cột order_id (PK):
  - Duy nhất: PASS
  - Không Null: PASS
Bảng SALES EMPLOYEE - Cột sales_employee_id (PK):
  - Duy nhất: PASS
  - Không Null: PASS

--- Kiểm tra Khóa Ngoại ---
FK sales_employee_id: PASS (Tất cả tham chiếu đều hợp lệ)

--- Kiểm tra Logic Khóa Phụ khác ---
Cột customer_id (FK dự kiến):
  - Không Null: PASS
  - Giá trị dương: PASS
Cột zip (FK dự kiến):
  - Không Null: PASS
  - Giá trị dương: PASS

=> TẤT CẢ KIỂM TRA HOÀN TẤT HỢP LỆ.


## 5. XUẤT BẢNG SILVER VÀ BÁO CÁO

In [37]:
import csv

# 4.1 Lưu 'sales_employee_df' ra file CSV
sales_employee_df.to_csv(
    "silver_sale_employee.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)
print("Đã lưu 'sales_employee.csv'")

# 4.2 Lưu 'order_df' ra file CSV
order_df.to_csv(
    "silver_orders.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)
print("Đã lưu 'order.csv'")

# 4.3 Báo cáo tóm tắt
print("\nBáo cáo tóm tắt:")
print(f"- Số dòng gốc của df: {len(df)}")
print(f"- Số dòng của sales_employee.csv: {len(sales_employee_df)}")
print(f"- Số dòng của order.csv: {len(order_df)}")

Đã lưu 'sales_employee.csv'
Đã lưu 'order.csv'

Báo cáo tóm tắt:
- Số dòng gốc của df: 646945
- Số dòng của sales_employee.csv: 200
- Số dòng của order.csv: 646945
